# Chapter 50: Advanced Vision Model Evaluation
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Menganalisis kerangka evaluasi komprehensif sistem visi komputer, menghitung kurva Precision-Recall dan Average Precision (AP) terinterpolasi, mengevaluasi reliabilitas probabilitas model menggunakan Expected Calibration Error (ECE), serta mensimulasikan perbaikan kalibrasi pasca-pelatihan dengan Temperature Scaling.

### 1. Landasan Teori & Formulasi Metrik Evaluasi
Metrik akurasi skalar tidak memadai pada sistem visi dunia nyata akibat ketidakseimbangan kelas (*class imbalance*) dan fenomena overconfidence.

All-Point Precision Interpolation:
$$P_{\text{interp}}(r) = \max_{\tilde{r} \ge r} P(\tilde{r}), \quad \text{AP} = \sum_{k=1}^n (R_k - R_{k-1}) P_{\text{interp}}(R_k)$$

Expected Calibration Error (ECE) pada $M$ bin interval keyakinan:
$$\text{ECE} = \sum_{m=1}^M \frac{|B_m|}{N} \left| \text{acc}(B_m) - \text{conf}(B_m) \right|$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy murni.")

In [ ]:
# 3. Implementasi Kurva Precision-Recall & All-Point Interpolated AP
def compute_interpolated_ap(confidences, is_tp, num_gt):
    sort_idx = np.argsort(-confidences)
    tp_sorted = is_tp[sort_idx]
    
    cum_tp = np.cumsum(tp_sorted)
    cum_fp = np.cumsum(1 - tp_sorted)
    
    recalls = cum_tp / num_gt
    precisions = cum_tp / (cum_tp + cum_fp)
    
    mrec = np.concatenate(([0.0], recalls))
    mpre = np.concatenate(([1.0], precisions))
    
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
        
    i_change = np.where(mrec[1:] != mrec[:-1])[0] + 1
    ap = float(np.sum((mrec[i_change] - mrec[i_change - 1]) * mpre[i_change]))
    return ap, recalls, precisions

# Data 10 deteksi uji sesuai Seksi 3
conf = np.array([0.95, 0.90, 0.82, 0.75, 0.68, 0.55, 0.45, 0.35, 0.25, 0.15])
tp = np.array([1, 1, 0, 1, 0, 1, 0, 0, 1, 0])
num_gt = 5

ap_score, recs, precs = compute_interpolated_ap(conf, tp, num_gt)
print(f"Hasil Average Precision (AP): {ap_score:.5f}")
np.testing.assert_allclose(ap_score, 0.79446, atol=1e-4)
print("VALIDASI BERHASIL: Solusi AP terinterpolasi presisi sesuai derivasi manual!")

In [ ]:
# 4. Evaluasi Kalibrasi: Expected Calibration Error (ECE) & Brier Score
def compute_ece(confidences, is_correct, bin_edges):
    n = len(confidences)
    ece = 0.0
    stats = []
    for i in range(len(bin_edges) - 1):
        low, high = bin_edges[i], bin_edges[i + 1]
        mask = (confidences >= low) & (confidences <= high) if i == 0 else (confidences > low) & (confidences <= high)
        count = np.sum(mask)
        if count > 0:
            acc = float(np.mean(is_correct[mask]))
            avg_c = float(np.mean(confidences[mask]))
            err = abs(acc - avg_c)
            ece += (count / n) * err
            stats.append((low, high, count, acc, avg_c, err))
    return float(ece), stats

bin_edges = np.array([0.0, 0.40, 0.70, 1.00])
ece_val, stats = compute_ece(conf, tp, bin_edges)
brier = float(np.mean((conf - tp)**2))

print(f"ECE (3 Bins): {ece_val:.4f}")
print(f"Brier Score:  {brier:.4f}")
np.testing.assert_allclose(ece_val, 0.1350, atol=1e-4)
print("VALIDASI BERHASIL: ECE cocok persis dengan perhitungan teoritis [0.1350]!")

In [ ]:
# 5. Kalibrasi Temperature Scaling & Profiling Latensi Inferensi
p_safe = np.clip(conf, 1e-7, 1.0 - 1e-7)
logits = np.log(p_safe / (1.0 - p_safe))

# Pelunakan logit dengan T = 1.3
T_opt = 1.3
p_cal = 1.0 / (1.0 + np.exp(-logits / T_opt))
ece_cal, _ = compute_ece(p_cal, tp, bin_edges)
brier_cal = float(np.mean((p_cal - tp)**2))

print(f"Sebelum Kalibrasi: ECE = {ece_val:.4f}, Brier = {brier:.4f}")
print(f"Sesudah Kalibrasi: ECE = {ece_cal:.4f}, Brier = {brier_cal:.4f} (T = {T_opt})")
assert ece_cal < ece_val, "Kalibrasi harus menurunkan ECE!"
assert brier_cal < brier, "Kalibrasi harus menurunkan Brier Score!"

# Simulasi Profiling Distribusi Latensi Inferensi Per-Frame
np.random.seed(42)
frame_latencies_ms = np.random.normal(loc=16.5, scale=2.0, size=500)
frame_latencies_ms = np.clip(frame_latencies_ms, 10.0, 45.0)

p50 = np.percentile(frame_latencies_ms, 50)
p95 = np.percentile(frame_latencies_ms, 95)
p99 = np.percentile(frame_latencies_ms, 99)
fps_equiv = 1000.0 / p50

print(f"\n--- Profil Latensi Edge Model ---")
print(f"P50 (Median): {p50:.2f} ms (~{fps_equiv:.1f} FPS)")
print(f"P95:          {p95:.2f} ms")
print(f"P99 (Tail):   {p99:.2f} ms")
print("VERIFIKASI SISTEM EVALUASI SELESAI: Lulus deterministik.")

### 6. Rangkuman & Pembahasan Akademik
1. **Evaluasi Holistik**: Kurva PR dan AP terinterpolasi memberikan penilaian objektif tanpa terdistorsi oleh ketidakseimbangan kelas.
2. **Pentingnya Kalibrasi**: Expected Calibration Error (ECE) mendeteksi overconfidence pada model modern; Temperature Scaling memperbaikinya tanpa mengorbankan akurasi klasifikasi.
3. **Metrik Operasional Edge**: Pencatatan profil latensi persentil (P50, P95, P99) menjamin keandalan deterministik sistem pada penerapan dunia nyata.